# Bénin Risk Map — Notebook d'analyse exploratoire

**Hackathon iSHEERO × DataCamp Donates 2026 · Team09**

Ce notebook constitue le livrable d'exploration de la Phase 1. Il :
1. Charge le snapshot GDELT 2025 enrichi (3 pays — Bénin, Burkina Faso, Niger).
2. Présente **6 visualisations commentées** sur les données du Bénin.
3. Entraîne / utilise un **modèle ML de sentiment multilingue** (xlm-roberta) pour valider le signal `AvgTone` de GDELT.
4. Conclut par 5 insights provisoires nourrissant le pitch.

Doctrine appliquée : `story-as-unit`, `confidence-as-slider`, `domains-not-codes`, `admin1-first` (cf. `docs/01_doctrine.md`).

## 1. Setup et chargement des données

Les données sont produites par le pipeline `make extract && make process`. Si vous obtenez `FileNotFoundError`, lancez d'abord ces commandes.

In [ ]:
import sys
from pathlib import Path

# Permettre l'import depuis src/ depuis le notebook
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.config import (
    BENIN_DEPARTMENTS,
    DOMAINS,
    FIPS_BENIN,
    FIPS_BURKINA,
    FIPS_NIGER,
    PROCESSED_DIR,
)
from src.analytics import aggregate, change_points, network_ops
from src.viz import maps, timeseries, network as viz_network

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 30)

In [ ]:
EVENTS_PATH = PROCESSED_DIR / "events_enriched.parquet"
STORIES_PATH = PROCESSED_DIR / "stories.parquet"

if not EVENTS_PATH.exists():
    raise FileNotFoundError(
        f"{EVENTS_PATH} introuvable.\n"
        "Exécuter : `make extract && make process` (depuis la racine du repo)."
    )

events = pd.read_parquet(EVENTS_PATH)
stories = pd.read_parquet(STORIES_PATH) if STORIES_PATH.exists() else pd.DataFrame()

print(f"Events     : {len(events):>8,} lignes")
print(f"Stories    : {len(stories):>8,} lignes")
print(f"Période    : {events['SQLDATE'].min().date()} → {events['SQLDATE'].max().date()}")
print(f"Pays       : {sorted(events['ActionGeo_CountryCode'].dropna().unique().tolist())}")
print(f"Domaines   : {sorted(events['risk_domain'].dropna().unique().tolist())}")
print(f"Confiance strict / large : {(events['confidence_tier']=='strict').sum():,} / {(events['confidence_tier']=='large').sum():,}")

## 2. Aperçu général

Distribution du volume d'events par pays et par domaine de risque, sur l'ensemble de la période. Premier signal sur le déséquilibre régional attendu.

In [ ]:
summary = (
    events.groupby(["ActionGeo_CountryCode", "risk_domain"])
    .size()
    .unstack(fill_value=0)
)
summary.loc["TOTAL"] = summary.sum()
summary["TOTAL"] = summary.sum(axis=1)
summary

## 3. Visualisation 1 — Volume hebdomadaire par pays

Permet de repérer les pics d'activité événementielle. Une attaque majeure ou une crise politique se traduit par un pic visible. La comparaison BN / UV / NG révèle le déséquilibre d'intensité régionale.

In [ ]:
country_labels = {FIPS_BENIN: "Bénin", FIPS_BURKINA: "Burkina Faso", FIPS_NIGER: "Niger"}
ts_by_country = {}
for code, label in country_labels.items():
    sub = events[events["ActionGeo_CountryCode"] == code]
    if not sub.empty:
        ts_by_country[label] = aggregate.time_series(sub, freq="W")

fig = timeseries.multi_line(
    ts_by_country,
    title="Events hebdomadaires — Bénin / Burkina Faso / Niger (2025)",
    yaxis_title="Events / semaine",
)
fig.show()

**Lecture** : la courbe attendue montre un Bénin plus calme en moyenne que le Burkina/Niger, mais avec une tendance pouvant être à la hausse au second semestre 2025 si le débordement sahélien se poursuit.

## 4. Visualisation 2 — Décomposition par domaine de risque (Bénin)

Application directe du principe `domains-not-codes` de la doctrine : on visualise le poids relatif des domaines (sécuritaire, politique, économique, humanitaire) plutôt que des codes CAMEO bruts.

In [ ]:
benin = events[events["ActionGeo_CountryCode"] == FIPS_BENIN]
by_domain = (
    benin.groupby("risk_domain").size().reset_index(name="n_events")
    .sort_values("n_events", ascending=False)
)
fig = px.bar(
    by_domain,
    x="risk_domain",
    y="n_events",
    title="Distribution des events béninois par domaine de risque",
    color="risk_domain",
)
fig.update_layout(showlegend=False)
fig.show()

**Lecture** : le profil attendu pour le Bénin penche vers `politique` en volume absolu, avec une part `sécuritaire` qui devrait croître au fil des trimestres si l'hypothèse de débordement sahélien est validée.

## 5. Visualisation 3 — Carte choroplèthe par département béninois

Application du principe `admin1-first`. La carte permet de répondre directement à Q1 : quels départements concentrent le risque sécuritaire ?

In [ ]:
benin_secu = benin[benin["risk_domain"] == "securitaire"]
by_dept = aggregate.by_admin1(benin_secu, value_col="n_events")
by_dept = by_dept.dropna(subset=["dept_normalized"]).sort_values("n_events", ascending=False)

fig = maps.choropleth_benin(
    by_dept,
    value_col="n_events",
    title="Risque sécuritaire par département (Bénin)",
)
fig.show()

**Note** : sans le geojson `data/external/benin_admin1.geojson`, la fonction renvoie un bar chart de fallback. Pour obtenir la carte, télécharger un shapefile officiel des départements et le convertir en geojson.

## 6. Visualisation 4 — Top 15 acteurs cités aux côtés du Bénin

Première porte d'entrée vers Q3 : qui parle de qui ? On filtre sur les acteurs typés (GOV, MIL, IGO, NGO) pour éviter le bruit (cf. doctrine, limite documentée des codes acteurs GDELT).

In [ ]:
from src.config import CAMEO_BENIN

# Events où Bénin apparaît comme actor1 ou actor2
benin_centric = events[
    (events["Actor1CountryCode"] == CAMEO_BENIN)
    | (events["Actor2CountryCode"] == CAMEO_BENIN)
]

# Extraire l'"autre" acteur (celui qui n'est pas le Bénin)
other_a = benin_centric.loc[benin_centric["Actor1CountryCode"] != CAMEO_BENIN, "Actor1Name"]
other_b = benin_centric.loc[benin_centric["Actor2CountryCode"] != CAMEO_BENIN, "Actor2Name"]
others = pd.concat([other_a, other_b]).dropna()

top_actors = others.value_counts().head(15).reset_index()
top_actors.columns = ["actor", "co_occurrences"]

fig = px.bar(
    top_actors.sort_values("co_occurrences"),
    x="co_occurrences",
    y="actor",
    orientation="h",
    title="Top 15 acteurs co-apparaissant avec le Bénin",
)
fig.show()

## 7. Visualisation 5 — Évolution du ton (AvgTone) du Bénin sur 2025

Préparation directe de Q2. On lisse le ton sur fenêtre 7 jours, pondéré par `NumMentions`. La détection de points de bascule sera faite plus loin via PELT.

In [ ]:
tone = change_points.smooth_tone(benin, window="7D")
breakpoints = change_points.detect_breakpoints(tone, penalty=8.0)

fig = timeseries.line_with_breakpoints(
    tone,
    breakpoints=breakpoints,
    title="Ton lissé du Bénin (AvgTone, fenêtre 7j) — ruptures détectées en pointillés",
    yaxis_title="AvgTone",
)
fig.show()

print(f"Ruptures détectées : {len(breakpoints)}")
for bp in breakpoints:
    print(f"  - {bp.date()}")

## 8. Visualisation 6 — Heatmap département × mois (sécuritaire, Bénin)

Croise les deux dimensions clés (où, quand) pour le risque sécuritaire.

In [ ]:
benin_secu = benin_secu.copy()
benin_secu["month"] = benin_secu["SQLDATE"].dt.to_period("M").astype(str)
heatmap_df = (
    benin_secu.dropna(subset=["dept_normalized"])
    .groupby(["dept_normalized", "month"])
    .size()
    .reset_index(name="n_events")
)

fig = px.density_heatmap(
    heatmap_df,
    x="month",
    y="dept_normalized",
    z="n_events",
    color_continuous_scale="Reds",
    title="Heatmap risque sécuritaire — département × mois (Bénin)",
)
fig.show()

## 9. Modèle ML — Sentiment multilingue (xlm-roberta) sur titres d'articles

On utilise le modèle `nlptown/bert-base-multilingual-uncased-sentiment` (HuggingFace, gratuit, 100% local). Objectif : valider sur sous-échantillon le signal `AvgTone` de GDELT (calculé par dictionnaire GCAM, donc grossier).

**Hypothèse** : si `AvgTone` est un signal valide, il devrait corréler positivement avec la polarité prédite par xlm-roberta sur les titres d'articles correspondants.

In [ ]:
from src.ml.sentiment import score_titles

# Sous-échantillon de 100 events béninois avec URL renseignée
sample = (
    benin.dropna(subset=["SOURCEURL"])
    .sample(n=min(100, len(benin)), random_state=42)
    .reset_index(drop=True)
)
# Pour le notebook on utilise l'URL comme proxy de titre (simplification)
# En production on récupère le vrai titre via GDELT DOC API ou newspaper3k
titles = sample["SOURCEURL"].astype(str).tolist()
scores = score_titles(titles, batch_size=8)
scores

In [ ]:
# Comparaison ton GDELT (AvgTone normalisé [-1, +1]) vs xlm-roberta (polarity [-1, +1])
merged = sample.copy()
merged["avgtone_norm"] = merged["AvgTone"] / 100.0
merged = merged.merge(
    scores.reset_index(drop=True)[["polarity"]],
    left_index=True,
    right_index=True,
)

correlation = merged[["avgtone_norm", "polarity"]].corr().iloc[0, 1]
print(f"Corrélation AvgTone (GDELT) vs polarity (xlm-roberta) : {correlation:.3f}")

fig = px.scatter(
    merged,
    x="avgtone_norm",
    y="polarity",
    title=f"Validation : AvgTone GDELT vs sentiment xlm-roberta (corr = {correlation:.3f})",
    trendline="ols",
)
fig.show()

**Interprétation attendue** : une corrélation > 0.3 valide qualitativement le signal `AvgTone`. Si elle est < 0.1, on signalera dans le rapport que `AvgTone` doit être pris avec prudence et préférera le modèle `xlm-roberta` pour les conclusions sur le ton.

## 10. Conclusion et insights provisoires

Cette exploration valide la faisabilité méthodologique des 3 questions de recherche du projet. Les chiffres ci-dessous sont des **placeholders** à remplacer par les valeurs réelles produites par les modules `src.questions.q1_terrain`, `q2_tone`, `q3_network`.

### Cinq insights provisoires (à valider sur les données réelles)

1. **Sous-couverture** — un event béninois génère **N×** moins de mentions médiatiques qu'un event burkinabè comparable.
2. **Carte du risque** — le risque sécuritaire opérationnel se concentre dans les départements frontaliers du nord (Atacora, Alibori), avec une intensité représentant **X%** de celle observée au Burkina Faso.
3. **Bascule narrative** — le ton mondial sur le Bénin connaît **N ruptures** sur 2025, dont la plus marquée coïncide avec **[story]**.
4. **Recomposition diplomatique** — depuis juillet 2023, la co-occurrence Bénin–Niger a chuté de **X%**, tandis que Bénin–[acteur] a doublé.
5. **Asymétrie de couverture** — le ton sécuritaire se dégrade continûment (-Y points) tandis que le ton économique reste stable autour de **+Z**.

### Lancer les questions complètes

```bash
make question Q=Q1
make question Q=Q2
make question Q=Q3
```